# Cultura Database — Getting Started

This notebook shows how to load and query the Cultura Database using Polars.

**Example**: Extract all scientists from contemporary China (1900–2024) and plot their number over time.

In [ ]:
import sqlite3
import polars as pl
import matplotlib.pyplot as plt

## 1. Connect to the Database

The database is located at `data/humans_clean.sqlite3`.

In [ ]:
DB_PATH = "data/humans_clean.sqlite3"
conn = sqlite3.connect(DB_PATH)

# Quick check: how many individuals in the database?
total = pl.read_database("SELECT COUNT(*) as n FROM individuals", conn)
print(f"Total individuals: {total['n'][0]:,}")

## 2. Extract Chinese Scientists (1900–2024)

We filter on:
- `nationalities_en` containing a Chinese nationality
- `birthdate` between 1900 and 2024

In [ ]:
query = """
SELECT wikidata_id, name_en, birthdate, occupations_en, nationalities_en
FROM individuals
WHERE (
    nationalities_en LIKE '%People\'s Republic of China%'
    OR nationalities_en LIKE '%Republic of China%'
    OR nationalities_en LIKE '%Chinese%'
)
AND birthdate IS NOT NULL
AND birthdate >= '1900'
AND birthdate < '2025'
"""

df = pl.read_database(query, conn)
print(f"Chinese individuals born 1900-2024: {len(df):,}")
df.head(10)

In [ ]:
# Parse birth year
df = df.with_columns(
    pl.col("birthdate").str.slice(0, 4).cast(pl.Int32, strict=False).alias("birth_year")
).filter(pl.col("birth_year").is_not_null())

# Filter to scientists (occupations containing science-related terms)
scientist_keywords = ["scientist", "physicist", "chemist", "biologist", "mathematician",
                      "engineer", "researcher", "astronomer", "geologist", "computer scientist"]

pattern = "|".join(scientist_keywords)
scientists = df.filter(
    pl.col("occupations_en").fill_null("").str.to_lowercase().str.contains(pattern)
)
print(f"Chinese scientists born 1900-2024: {len(scientists):,}")

## 3. Plot the Number of Scientists by Decade

In [ ]:
by_decade = (
    scientists
    .with_columns((pl.col("birth_year") // 10 * 10).alias("decade"))
    .group_by("decade")
    .agg(pl.len().alias("count"))
    .sort("decade")
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(by_decade["decade"].to_list(), by_decade["count"].to_list(), width=8, color="#c0392b", edgecolor="white")
ax.set_xlabel("Birth Decade")
ax.set_ylabel("Number of Scientists")
ax.set_title("Scientists from China in the Cultura Database (born 1900–2024)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Explore Historical Polities

Query individuals linked to historical polities via Cliopatria.

In [ ]:
# Top 10 polities by number of individuals
top_polities = pl.read_database("""
    SELECT polity_name, COUNT(*) as n
    FROM individuals_cliopatria
    GROUP BY polity_name
    ORDER BY n DESC
    LIMIT 10
""", conn)
print("Top 10 historical polities:")
top_polities

In [ ]:
# Writers in the Ottoman Empire
ottoman_writers = pl.read_database("""
    SELECT ic.name_en, ic.impact_date, ic.polity_name
    FROM individuals_cliopatria ic
    JOIN individuals i ON ic.wikidata_id = i.wikidata_id
    WHERE ic.polity_name = 'Ottoman Empire'
      AND i.occupations_en LIKE '%writer%'
    ORDER BY ic.impact_date
    LIMIT 20
""", conn)
print("Writers in the Ottoman Empire:")
ottoman_writers

## 5. Explore Occupations

In [ ]:
# Top 20 occupations
top_occupations = pl.read_database("""
    SELECT name_en, count, meta_occupation
    FROM occupations
    ORDER BY count DESC
    LIMIT 20
""", conn)
print("Top 20 occupations:")
top_occupations

## 6. Explore Regions

In [ ]:
# Individuals by macro-region
by_macro_region = pl.read_database("""
    SELECT macro_region, COUNT(*) as n
    FROM individuals_regions
    GROUP BY macro_region
    ORDER BY n DESC
""", conn)
print("Individuals by macro-region:")
by_macro_region

In [ ]:
conn.close()